# 02 · Estrategias sin modelo en validación

Usamos exclusivamente `data/splits/validacion.csv`. Este notebook no carga
el CSV limpio completo ni train ni test, y no entrena modelos.

- Positivo: `Churn = Yes`; la decisión asociada es **contactar**.
- Negativo: `Churn = No`; la decisión asociada es **no contactar**.
- Costo: **120 × FN + 15 × FP**, en USD. TP y TN tienen costo cero en este ejercicio.

Comparamos dos decisiones constantes: no contactar a nadie y contactar a todos.


In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

RAIZ = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "CONTEXT.md").is_file() and (p / "src" / "limpieza.py").is_file()
)
import numpy as np
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score, recall_score,
)

SPLITS = RAIZ / "data" / "splits"
RUTA_VALIDACION = SPLITS / "validacion.csv"
metadata = json.loads((SPLITS / "metadata.json").read_text(encoding="utf-8"))
huella = hashlib.sha256(RUTA_VALIDACION.read_bytes()).hexdigest()
if huella != metadata["archivos_sha256"]["validacion.csv"]:
    raise ValueError("Validación no coincide con la separación persistida.")

validacion = pd.read_csv(RUTA_VALIDACION)
assert len(validacion) == metadata["filas"]["validacion"]
assert validacion["Churn"].notna().all()
assert set(validacion["Churn"]) <= {"No", "Yes"}
y_validacion = validacion["Churn"].map({"No": 0, "Yes": 1}).to_numpy()
COSTO_FN = 120
COSTO_FP = 15


## Matrices y métricas

Las matrices tienen **filas = clase real [No, Yes]** y
**columnas = decisión [no contactar, contactar]**:
`[[TN, FP], [FN, TP]]`.

Accuracy = (TP + TN) / N; precision = TP / (TP + FP);
recall = TP / (TP + FN).
Para “no contactar a nadie”, precision no está definida porque no hay
predicciones positivas. Se registra **0 por convención** (`zero_division=0`).


In [2]:
estrategias = {
    "No contactar a nadie": np.zeros(len(y_validacion), dtype=int),
    "Contactar a todos": np.ones(len(y_validacion), dtype=int),
}
resultados = []
for nombre, prediccion in estrategias.items():
    matriz = confusion_matrix(y_validacion, prediccion, labels=[0, 1])
    tn, fp, fn, tp = (int(valor) for valor in matriz.ravel())
    print(nombre)
    display(pd.DataFrame(
        matriz,
        index=pd.Index(["No", "Yes"], name="Churn real"),
        columns=pd.Index(["No contactar", "Contactar"], name="Decisión"),
    ))
    resultados.append({
        "estrategia": nombre,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "accuracy": accuracy_score(y_validacion, prediccion),
        "precision": precision_score(y_validacion, prediccion, zero_division=0),
        "recall": recall_score(y_validacion, prediccion, zero_division=0),
        "costo_total_usd": COSTO_FN * fn + COSTO_FP * fp,
    })
resumen = pd.DataFrame(resultados).set_index("estrategia")
display(resumen)


No contactar a nadie


Decisión,No contactar,Contactar
Churn real,,
No,776,0
Yes,280,0


Contactar a todos


Decisión,No contactar,Contactar
Churn real,,
No,0,776
Yes,0,280


,TN,FP,FN,TP,accuracy,precision,recall,costo_total_usd
estrategia,,,,,,,,
No contactar a nadie,776,0,280,0,0.734848,0.000000,0.0,33600
Contactar a todos,0,776,0,280,0.265152,0.265152,1.0,11640


In [3]:
SALIDA = RAIZ / "outputs" / "baselines_validacion"
SALIDA.mkdir(parents=True, exist_ok=True)
resumen.to_csv(SALIDA / "metricas.csv", lineterminator="\n")
(SALIDA / "metadata.json").write_text(json.dumps({
    "conjunto": "validacion",
    "validacion_sha256": huella,
    "costo_fn_usd": COSTO_FN,
    "costo_fp_usd": COSTO_FP,
    "clase_positiva": "Churn=Yes; decisión=contactar",
    "matriz": "filas reales [No, Yes]; columnas decisiones [No contactar, Contactar]",
    "precision_sin_predicciones_positivas": "0 por convención (zero_division=0)",
}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

menor_costo = resumen["costo_total_usd"].idxmin()
mayor_accuracy = resumen["accuracy"].idxmax()
ahorro = resumen["costo_total_usd"].max() - resumen["costo_total_usd"].min()
print(f"Menor costo en validación: {menor_costo}.")
print(f"Mayor accuracy en validación: {mayor_accuracy}.")
print(f"Diferencia de costo entre estrategias: USD {ahorro:,.0f}.")
print("No se ha cargado ni evaluado el test.")


Menor costo en validación: Contactar a todos.
Mayor accuracy en validación: No contactar a nadie.
Diferencia de costo entre estrategias: USD 21,960.
No se ha cargado ni evaluado el test.


## Interpretación

La estrategia con mayor accuracy puede tener mayor costo: perder un cliente
cuesta ocho veces más que contactar innecesariamente a uno que permanece.
Estos resultados son referencias mínimas para comparar futuros modelos en
**la misma validación**.

Los costos son supuestos del ejercicio. No se ha estimado que contactar a un
cliente evite su abandono, ni el efecto de una campaña de retención.
